[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo07.ipynb)

# MATH840 — Week 7 practice: ARIMA by hand, four times

**This session is not marked.** One small toolkit, written once, and used on four series that each
behave differently. The recipe is the one from the *ARIMA Models, Part 1* lecture, and it is the recipe
for tonight's lab on your own series:

1. **difference** until stationary — `nsdiffs`, then `ndiffs`;
2. **read** the ACF and PACF of the differenced series and write down candidates;
3. **fit** them, check the residuals with the right `model_df`, compare with `AutoARIMA`;
4. **decide** on a rolling origin, with a benchmark in the table.

| Example | Series | What it teaches | Minutes |
|---|---|---|---|
| 0 | — | the toolkit | 10 |
| 1 | Gabon's exports, annual | the textbook case: the ACF and PACF give the model away | 15 |
| 2 | US egg prices, annual | ARIMA's honest answer is "a random walk" — and then the benchmark is the model | 10 |
| 3 | US construction employment, monthly | **your case**: a seasonal difference, then an ARMA | 25 |
| 4 | Australian gas, quarterly | where part 1's tools stop, and what part 2 adds | 15 |

None of the four series is in the Track B pool.

## 0. Setup and the toolkit

In [ ]:
!pip install -q statsforecast==2.1.1

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import warnings
warnings.filterwarnings("ignore")
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import (ARIMA, AutoARIMA, AutoETS, HistoricAverage, Naive,
                                  RandomWalkWithDrift, SeasonalNaive)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase

plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

Four functions, one per step of the recipe. Every example below calls the same four — that is the
point: the decisions change from series to series, the code does not.

In [ ]:
def difference_plan(y, m):
    # Step 1: D from seasonal strength, then d from KPSS on what is left. Returns D, d and the differenced series.
    D = nsdiffs(y, period=m) if m > 1 else 0
    z = y[m:] - y[:-m] if D else y.copy()
    d = ndiffs(z)
    z = np.diff(z, n=d) if d else z
    print(f"D = {D}, d = {d}: {len(y)} observations -> {len(z)} after differencing")
    return D, d, z


def acf_pacf(z, lags, title=""):
    # Step 2: the two plots you read p and q (and later P and Q) from.
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    plot_acf(z, lags=lags, zero=False, ax=axes[0], auto_ylims=True, title=f"{title} ACF")
    plot_pacf(z, lags=lags, zero=False, ax=axes[1], auto_ylims=True, title=f"{title} PACF")
    plt.show()


def compare(y, candidates, m=1, lb_lags=10):
    # Step 3: fit each candidate on the whole history; AICc and Ljung-Box with model_df = p + q + P + Q.
    rows = []
    for name, model in candidates.items():
        fit = model.fit(y=y)
        p, q, P, Q = fit.model_["arma"][:4]
        resid = fit.model_["residuals"]
        rows.append({"candidate": name,
                     "model": str(ARIMASummary(fit.model_)).strip(),
                     "AICc": fit.model_["aicc"],
                     "Ljung-Box p": acorr_ljungbox(resid, lags=[lb_lags], model_df=p + q + P + Q)["lb_pvalue"].iloc[0],
                     f"resid acf({m})": pd.Series(resid).autocorr(m) if m > 1 else np.nan})
    return pd.DataFrame(rows).dropna(axis=1, how="all")


def rolling(df, models, h, m, windows=5, step=None, freq=None):
    # Step 4: MASE over `windows` origins, h steps ahead each, every model on the same windows.
    cv = StatsForecast(models=models, freq=freq or pd.infer_freq(df["ds"])).cross_validation(
        df=df, h=h, n_windows=windows, step_size=step or h)
    out = evaluate(cv.drop(columns="cutoff"), metrics=[partial(mase, seasonality=m)], train_df=df)
    return out.drop(columns=["unique_id", "metric"]).T.rename(columns={0: "MASE"}).sort_values("MASE")

In [ ]:
def forecast_plot(df, model, h, title, last=60):
    # The chosen model refitted on everything, with its 80% and 95% intervals.
    fc = model.fit(y=df["y"].to_numpy(float)).predict(h=h, level=[80, 95])
    future = pd.date_range(df["ds"].iloc[-1], periods=h + 1, freq=pd.infer_freq(df["ds"]))[1:]
    fig, ax = plt.subplots()
    ax.plot(df["ds"].iloc[-last:], df["y"].iloc[-last:], color="#314f4f", linewidth=1, label="history")
    ax.plot(future, fc["mean"], color="#e64173", linewidth=2, label="forecast")
    for level, alpha in [(95, 0.12), (80, 0.25)]:
        ax.fill_between(future, fc[f"lo-{level}"], fc[f"hi-{level}"], color="#e64173", alpha=alpha,
                        label=f"{level}%")
    ax.legend(loc="upper left")
    ax.set_title(title)
    plt.show()

## 1. Gabon's exports: the textbook case

Exports as a share of GDP, 1960–2017. Annual, so no season: $m = 1$, and the question is only $d$, $p$
and $q$.

In [ ]:
economy = pd.read_csv(f"{BASE}/global_economy.csv", parse_dates=["ds"])
gabon = (economy.query("unique_id == 'Gabon'")[["unique_id", "ds", "Exports"]]
         .rename(columns={"Exports": "y"}).dropna().reset_index(drop=True))
y1 = gabon["y"].to_numpy(float)

fig, ax = plt.subplots()
ax.plot(gabon["ds"], y1, color="#314f4f")
ax.set_title("Gabon: exports, % of GDP")
plt.show()

D1, d1, z1 = difference_plan(y1, m=1)
acf_pacf(z1, lags=16, title="Gabon exports:")

::: Discussion
**Before the next cell: which model do the two plots name, and which lag in which plot tells you?**

Then: the series spends years above its average and years below it. Does that make it
non-stationary? What did KPSS say?
:::

In [ ]:
candidates = {
    "by hand: AR(1)": ARIMA(order=(1, 0, 0)),
    "alternative: MA(2)": ARIMA(order=(0, 0, 2)),
    "AutoARIMA": AutoARIMA(),
}
compare(y1, candidates)

In [ ]:
rolling(gabon,
        [ARIMA(order=(1, 0, 0), alias="AR(1) by hand"), AutoARIMA(alias="AutoARIMA"),
         Naive(alias="naive"), HistoricAverage(alias="mean"), AutoETS(season_length=1, alias="ETS")],
        h=5, m=1, windows=8, step=1)

In [ ]:
forecast_plot(gabon, ARIMA(order=(1, 0, 0)), h=10, title="Gabon exports: AR(1), ten years ahead")

::: Discussion
**The fixed AR(1) forecasts better on the rolling origin than `AutoARIMA` — even though on the full
history `AutoARIMA` chose exactly AR(1). How can that be?** (Hint: what does `AutoARIMA` do on each of
the eight shorter histories?)

And read the forecast: where does it go, and how fast? What in the AR coefficient decides that?
:::

## 2. US egg prices: when the honest answer is "random walk"

Price of a dozen eggs in constant dollars, 1900–1993.

In [ ]:
eggs = pd.read_csv(f"{BASE}/egg_prices.csv", parse_dates=["ds"])[["unique_id", "ds", "y"]]
y2 = eggs["y"].to_numpy(float)

fig, ax = plt.subplots()
ax.plot(eggs["ds"], y2, color="#314f4f")
ax.set_title("US egg prices, constant dollars")
plt.show()

D2, d2, z2 = difference_plan(y2, m=1)
acf_pacf(z2, lags=16, title="egg prices, first difference:")

In [ ]:
compare(y2, {"random walk: (0,1,0)": ARIMA(order=(0, 1, 0)),
             "MA(1)": ARIMA(order=(0, 1, 1)),
             "AR(1)": ARIMA(order=(1, 1, 0)),
             "AutoARIMA": AutoARIMA()})

In [ ]:
rolling(eggs,
        [AutoARIMA(alias="AutoARIMA"), Naive(alias="naive"),
         RandomWalkWithDrift(alias="drift"), AutoETS(season_length=1, alias="ETS")],
        h=5, m=1, windows=8, step=1)

::: Discussion
**The ACF and PACF of the changes show nothing, and `AutoARIMA` lands on ARIMA(0,1,0). Has the
modelling failed?**

What you are looking at is a result: the changes in egg prices are unpredictable from their own past,
so the best ARIMA *is* the naive forecast. `AutoARIMA` here is `naive` with extra steps — and `drift`,
the Week 1 benchmark that is ARIMA(0,1,0) **with** a constant, does slightly better.

If your own series lands here, what goes in the justification block?
:::

## 3. US construction employment: your case

Monthly, seasonal, 1990–2019 — the shape of most of your Track B series. Part 1's route for seasonal
data: **a seasonal difference, then a non-seasonal ARMA** on what is left.

In [ ]:
employment = pd.read_csv(f"{BASE}/us_employment.csv", parse_dates=["ds"])
construction = (employment.query("unique_id == 'Construction' and ds.dt.year >= 1990")
                .reset_index(drop=True))
y3 = construction["y"].to_numpy(float)

fig, ax = plt.subplots()
ax.plot(construction["ds"], y3, color="#314f4f")
ax.set_title("US construction employment, thousands")
plt.show()

D3, d3, z3 = difference_plan(y3, m=12)

In [ ]:
fig, ax = plt.subplots()
ax.plot(construction["ds"].iloc[12:], z3, color="#314f4f")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("After the seasonal difference: the change on the same month last year")
plt.show()
acf_pacf(z3, lags=36, title="seasonal difference:")

::: Discussion
**The ACF decays very slowly, yet KPSS says $d = 0$. Would you difference again?**

Two defensible answers, and they lead to different models. AICc cannot settle it — the two are fitted
to different data. Write both down; the rolling origin below decides.

And read the PACF: one spike towers over everything else. Which lag, how large, and what model does
that name for the $d = 0$ reading?
:::

In [ ]:
seasonal = partial(ARIMA, season_length=12)
candidates = {
    "d=0 by hand: AR(1)": seasonal(order=(1, 0, 0), seasonal_order=(0, 1, 0), include_drift=True),
    "d=1 by hand: (0,1,0)": seasonal(order=(0, 1, 0), seasonal_order=(0, 1, 0)),
    "AutoARIMA, no P or Q": AutoARIMA(season_length=12, max_P=0, max_Q=0),
}
print("AR(1) coefficient of the d=0 reading:",
      round(float(candidates["d=0 by hand: AR(1)"].fit(y=y3).model_["coef"]["ar1"]), 3))
compare(y3, candidates, m=12, lb_lags=24)

Two things to read here. The AR(1) coefficient is about **0.99**: an AR(1) that close to 1 *is* almost a
difference, so the $d = 0$ and $d = 1$ readings are nearly the same model — the debate above matters
less than it looked.

And the last column: a residual autocorrelation between $-0.2$ and $-0.4$ at lag 12, in every candidate. That is
the over-differencing signature from the lecture, one season back — the seasonal difference took out a
little too much, and Ljung-Box notices.

In [ ]:
rolling(construction,
        [seasonal(order=(1, 0, 0), seasonal_order=(0, 1, 0), include_drift=True, alias="d=0 by hand: AR(1)"),
         seasonal(order=(0, 1, 0), seasonal_order=(0, 1, 0), alias="d=1 by hand: (0,1,0)"),
         AutoARIMA(season_length=12, max_P=0, max_Q=0, alias="AutoARIMA, no P or Q"),
         AutoARIMA(season_length=12, alias="AutoARIMA with P, Q (part 2)"),
         AutoETS(season_length=12, alias="ETS"),
         SeasonalNaive(season_length=12, alias="snaive")],
        h=12, m=12, windows=5)

::: Discussion
**Ljung-Box rejected every candidate — and the same models beat seasonal naive by a factor of five.
The two hand readings tie, and both beat `AutoARIMA`. Which result goes into your report, and what does
each one tell you?**

The lag-12 residual is exactly what the *Part 2* lecture's seasonal MA term is for. Here the model with
it (`AutoARIMA with P, Q`) does **not** forecast better on this rolling origin. A better residual check is
not a promise of a better forecast — only the rolling origin can say which you got.
:::

In [ ]:
forecast_plot(construction, AutoARIMA(season_length=12, max_P=0, max_Q=0), h=24,
              title="US construction employment: seasonal difference + ARMA, two years ahead")

## 4. Australian gas: where part 1's tools stop

The quarterly gas production from Week 6's practice. Its seasonal swing grows with the level — ETS
handled that with a **multiplicative** season. ARIMA has no multiplicative season, so the same job is
done by a **log**: a swing that is a constant *share* of the level becomes a constant *size*.

In [ ]:
production = pd.read_csv(f"{BASE}/aus_production_long.csv", parse_dates=["ds"])
gas = (production.query("unique_id == 'Gas'")[["unique_id", "ds", "y"]]
       .dropna().sort_values("ds").reset_index(drop=True))
log_gas = gas.assign(y=np.log(gas["y"]))
y4 = log_gas["y"].to_numpy(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(gas["ds"], gas["y"], color="#314f4f")
axes[0].set_title("Gas production: the swing grows with the level")
axes[1].plot(gas["ds"], y4, color="#20B2AA")
axes[1].set_title("log(gas): the swing is roughly constant")
plt.show()

D4, d4, z4 = difference_plan(y4, m=4)
acf_pacf(z4, lags=16, title="log gas, seasonal + first difference:")

::: Discussion
**The biggest spike in both plots is at lag 4 — the seasonal lag — and it is negative. What would
part 1's route, a non-seasonal ARMA on this, have to do to model a lag-4 spike?**
:::

In [ ]:
compare(y4, {"AutoARIMA, no P or Q": AutoARIMA(season_length=4, max_P=0, max_Q=0),
             "AutoARIMA with P, Q (part 2)": AutoARIMA(season_length=4)},
        m=4, lb_lags=8)

In [ ]:
rolling(log_gas,
        [AutoARIMA(season_length=4, max_P=0, max_Q=0, alias="AutoARIMA, no P or Q"),
         AutoARIMA(season_length=4, alias="AutoARIMA with P, Q (part 2)"),
         AutoETS(season_length=4, alias="ETS on log"),
         SeasonalNaive(season_length=4, alias="snaive")],
        h=8, m=4, windows=5)

::: Discussion
**Here the seasonal difference + ARMA loses to seasonal naive, and the model with seasonal terms wins
clearly. In example 3 it was the other way round. What is different about the two series?**

Two things worth saying out loud: the lag-4 spike is large here (about $-0.5$) where the lag-12 one in
construction was moderate; and a non-seasonal ARMA can only reach lag 4 by spending four AR or MA
terms on it — `AutoARIMA` without $P, Q$ picked ARIMA(3,1,3) and still left it behind.

The scores are on the log scale. To forecast gas itself, exponentiate the forecast and the interval
bounds — the point forecast then becomes a median, not a mean (FPP 5.6).
:::

## What carries over to tonight

| Example | Hand reading | What decided |
|---|---|---|
| Gabon | AR(1) from one PACF spike | rolling origin: the fixed AR(1) beat `AutoARIMA` |
| Egg prices | nothing to read | the benchmark — `drift` *is* the model |
| Construction | $D = 1$, then AR(1) with $\phi \approx 0.99$ — or $d = 1$, the same thing | rolling origin, against a residual check that rejected |
| Gas | log, $D = 1$, $d = 1$, a lag-4 spike | the limit of part 1: seasonal terms win |

On your series:

1. `difference_plan(y, M)` — show $D$ and $d$, and the differenced series;
2. `acf_pacf(z, 3 * M)` — name the lags you read $p$ and $q$ from;
3. `compare(...)` — your candidates **and** `AutoARIMA`, with AICc and Ljung-Box;
4. `rolling(...)` — the decision, with your benchmark in the table.

The same four steps are in your
[Lab 6 template](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_lab07.ipynb),
with one rule built in: **no seasonal AR or MA terms** ($P = Q = 0$) — those are Lab 7.

**This week accuracy is measured, not marked.** Example 4 is why: with $P = Q = 0$ whether you beat the
benchmark depends on how strong your series' seasonal autocorrelation is. If your series behaves like gas,
say so in your report, with the lag-4 (or lag-12) evidence — that is an argument, and the rubric pays for it.